# Geometric Brownian Motion Demo

Translated from QMCJu's GBM/gbm_demo.ipynb

Demonstrates GeometricBrownianMotion for financial modeling:
  S(t) = S₀ exp[(γ - σ²/2) t + σ W(t)]

where W(t) is a standard Brownian motion.

In [1]:
using QMCJu
using Statistics
using Printf

Basic GBM Sample Generation

In [2]:
println("="^60)
println("GeometricBrownianMotion: Basic Usage")
println("="^60)

d = 4           # 4 time steps
S0 = 100.0      # initial price
γ = 0.05        # drift (5% annual return)
σ2 = 0.04       # diffusion (volatility² = 20% vol)

dd = DigitalNetB2(d; seed=7)
gbm = GeometricBrownianMotion(dd;
    t_final=1.0, initial_value=S0, drift=γ, diffusion=σ2)

println("  $gbm")
println("  Time vector: t = [$(join([@sprintf("%.2f", t) for t in gbm.time_vector], ", "))]")
println()

n = 10_000
x = gen_samples(dd, n)
paths = transform(gbm, x)

println("  Generated $n sample paths")
println("  Path shape: $(size(paths))")
println()

GeometricBrownianMotion: Basic Usage
  GeometricBrownianMotion(d=4, S₀=100.0, γ=0.05, σ²=0.04)
  Time vector: t = [0.25, 0.50, 0.75, 1.00]

  Generated 10000 sample paths
  Path shape: (10000, 4)



Show a few sample paths

In [3]:
println("  First 5 paths (S₀, S(t₁), S(t₂), S(t₃), S(T)):")
for i in 1:5
    @printf("    Path %d: %6.2f → [%6.2f  %6.2f  %6.2f  %6.2f]\n",
            i, S0, paths[i,1], paths[i,2], paths[i,3], paths[i,4])
end
println()

  First 5 paths (S₀, S(t₁), S(t₂), S(t₃), S(T)):
    Path 1: 100.00 → [104.73  106.46   93.37   88.05]
    Path 2: 100.00 → [122.43  105.18  109.80  134.36]
    Path 3: 100.00 → [ 94.01   99.26  108.75  110.89]
    Path 4: 100.00 → [ 92.84   92.69   74.45   80.33]
    Path 5: 100.00 → [ 98.67   97.97  109.12   95.50]



Statistical Properties

In [4]:
println("="^60)
println("Statistical Properties of GBM")
println("="^60)

Statistical Properties of GBM


E[S(t)] = S₀ exp(γ t)

In [5]:
for j in 1:d
    t = gbm.time_vector[j]
    exact_mean = S0 * exp(γ * t)
    empirical_mean = mean(paths[:, j])
    @printf("  t = %.2f: E[S(t)] = %.2f (exact = %.2f, err = %.2f%%)\n",
            t, empirical_mean, exact_mean,
            100.0 * abs(empirical_mean - exact_mean) / exact_mean)
end
println()

  t = 0.25: E[S(t)] = 101.26 (exact = 101.26, err = 0.00%)
  t = 0.50: E[S(t)] = 102.54 (exact = 102.53, err = 0.01%)
  t = 0.75: E[S(t)] = 103.83 (exact = 103.82, err = 0.01%)
  t = 1.00: E[S(t)] = 105.12 (exact = 105.13, err = 0.01%)



GBM values must be positive

In [6]:
@printf("  All values positive: %s (min = %.4f)\n",
        all(paths .> 0) ? "YES" : "NO", minimum(paths))
println()

  All values positive: YES (min = 49.3673)



GBM with FinancialOption

In [7]:
println("="^60)
println("GBM + FinancialOption Integration Pipeline")
println("="^60)

d_opt = 52
dd_opt = IIDStdUniform(d_opt; seed=7)

GBM + FinancialOption Integration Pipeline


IIDStdUniform(d=52)

Using BrownianMotion (internal GBM construction in FinancialOption)

In [8]:
tm_bm = BrownianMotion(dd_opt)
f_bm = FinancialOption(tm_bm;
    option_type=:asian, call_put=:call, mean_type=:arithmetic,
    volatility=0.2, start_price=100.0, strike_price=100.0,
    interest_rate=0.05)
sc_bm = CubMCCLT(f_bm; abs_tol=0.1)
result_bm = integrate(sc_bm)
@printf("  Asian Call (via BM):  %.4f  (n = %d)\n", result_bm.solution, result_bm.data[:n])

  Asian Call (via BM):  5.8317  (n = 63998)


Using GeometricBrownianMotion directly

In [9]:
dd_opt2 = IIDStdUniform(d_opt; seed=7)
gbm_tm = GeometricBrownianMotion(dd_opt2;
    t_final=1.0, initial_value=100.0, drift=0.05, diffusion=0.04)
f_gbm = FinancialOption(gbm_tm;
    option_type=:asian, call_put=:call, mean_type=:arithmetic,
    volatility=0.2, start_price=100.0, strike_price=100.0,
    interest_rate=0.05)
sc_gbm = CubMCCLT(f_gbm; abs_tol=0.1)
result_gbm = integrate(sc_gbm)
@printf("  Asian Call (via GBM): %.4f  (n = %d)\n", result_gbm.solution, result_gbm.data[:n])
println()

  Asian Call (via GBM): 5.8267  (n = 63992)



Different Volatilities

In [10]:
println("="^60)
println("European Call Prices for Different Volatilities")
println("="^60)

for vol in [0.1, 0.2, 0.3, 0.5, 0.8]
    dd_v = IIDStdUniform(52; seed=7)
    tm_v = BrownianMotion(dd_v)
    f_v = FinancialOption(tm_v;
        option_type=:european, call_put=:call,
        volatility=vol, start_price=100.0, strike_price=100.0,
        interest_rate=0.05)
    sc_v = CubMCCLT(f_v; abs_tol=5.0, n_init=4096)
    result_v = integrate(sc_v)
    @printf("  σ = %.1f: European Call = %.2f\n", vol, result_v.solution)
end
println()

println("="^60)
println("GBM demo completed!")

European Call Prices for Different Volatilities
  σ = 0.1: European Call = 6.77
  σ = 0.2: European Call = 10.34
  σ = 0.3: European Call = 14.04
  σ = 0.5: European Call = 21.46
  σ = 0.8: European Call = 32.40

GBM demo completed!
